# extensions

> User Python that registers more tools, skills, commands and hooks.

leela has done this once already. `leela.inspect.agent.load_inspectors` runs
`~/.config/leela/inspectors.py` through `runpy` and appends whatever it finds to the AST
policy -- an extension point, one hook deep. So extensions here are deliberately the *same
shape and the same directory*, because someone who has written an inspector already knows
the idiom and should not have to learn a second one.

An extension is a Python file defining `setup(ext)`:

    def setup(ext):
        @ext.tool
        def count_todos(path: str) -> str:
            "Count TODO comments in a file."
            return str(ext.host.read(path).count('TODO'))

        ext.skill('house-style', open('STYLE.md').read(), 'How we write code here')
        ext.on('after_turn', lambda agent: agent.host.note(f'{agent.use}'))

Project extensions are **off by default**, and that is not caution theatre: a file in
`.leela/extensions/` of a repository you cloned five minutes ago runs arbitrary Python
with the agent's tools in scope. Tau makes the same call, behind the same kind of flag.

Nothing here fails loudly. An extension that raises is reported in `notes` and skipped,
because the alternative is an IDE that will not open because of a stray file in a config
directory.


In [ ]:
#| default_exp extensions

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import runpy
from pathlib import Path
from ramabana.core import agent_err

In [ ]:
#| export
# The events an extension can hook. Small on purpose: both backends already have rich
# callback systems, and this is the seam for the *harness's* lifecycle, not the engine's.
EVENTS = ('before_turn', 'after_turn', 'before_tool', 'after_tool', 'compact', 'approval')

In [ ]:
#| export
class Registry:
    """What `setup(ext)` is handed: everything an extension may add, and nothing else.

    `host` and `agent` are exposed because an extension that cannot read a file or see the
    conversation is not worth writing. What is deliberately *not* here is any way to reach
    a backend's internals -- an extension that pokes at a litert conversation would break
    on the next model switch, and would break silently.
    """

    def __init__(self, host=None, agent=None):
        self.host, self.agent = host, agent
        self.tools, self.skills, self.commands = [], [], {}
        self.hooks = {e: [] for e in EVENTS}
        self.approve = None
        self.notes = []          # one line per extension: loaded, or why not

    # -- registration --------------------------------------------------------
    def tool(self, f):
        """Add a tool. Usable as a decorator.

        The contract is the backends' own: a plain function with type hints and a
        docstring. That docstring is what the model reads, so it is documentation and not
        a comment.
        """
        self.tools.append(f)
        return f

    def skill(self, name, text, description=''):
        "Add a skill the discovery pass would not find -- a file, a string, anything callable."
        from ramabana.skills import Skill, _describe
        s = Skill(name=name, source='ext', description=description or _describe(text if isinstance(text, str) else ''),
                  where='extension', _text=text)
        self.skills.append(s)
        return s

    def command(self, name, fn, help=''):
        "Add a slash command. `fn(agent, arg)` returns text for the frontend to show."
        self.commands[name.lstrip('/')] = (fn, help)
        return fn

    def on(self, event, fn):
        "Hook a harness lifecycle event. Unknown event names are an error, not a silent no-op."
        if event not in EVENTS: raise KeyError(f'unknown event {event!r}; known: {", ".join(EVENTS)}')
        self.hooks[event].append(fn)
        return fn

    def approval(self, fn):
        "Replace the approval policy wholesale. The last extension to call this wins."
        self.approve = fn
        return fn

    # -- dispatch ------------------------------------------------------------
    def fire(self, event, *args, **kw):
        "Run every hook for `event`, swallowing failures. Returns how many ran cleanly."
        n = 0
        for f in self.hooks.get(event, ()):
            try: f(*args, **kw); n += 1
            except Exception as e: self.notes.append(f'{event} hook failed: {agent_err(e)}')
        return n

In [ ]:
#| export
def ext_dirs(roots=(), cfg=None, project=False):
    "Where extensions are looked for. Project directories only when explicitly allowed."
    ds = []
    if cfg is not None: ds.append(Path(cfg)/'extensions')
    if project:
        for r in roots: ds.append(Path(r)/'.leela'/'extensions')
    return ds

In [ ]:
#| export
def load(reg, roots=(), cfg=None, project=False, paths=()):
    """Run every extension found, calling its `setup(reg)`. Returns the registry.

    A file with no `setup` is loaded and left alone rather than reported as broken: that is
    how a shared helper module sitting in the same directory should behave.
    """
    files = []
    for d in ext_dirs(roots, cfg, project):
        if Path(d).is_dir(): files += sorted(p for p in Path(d).glob('*.py') if not p.name.startswith('_'))
    for p in paths or ():
        p = Path(p)
        files += sorted(p.glob('*.py')) if p.is_dir() else [p]
    for f in files:
        try:
            ns = runpy.run_path(str(f))
        except Exception as e:
            reg.notes.append(f'{f.name}: failed to load ({agent_err(e)})')
            continue
        fn = ns.get('setup')
        if not callable(fn):
            reg.notes.append(f'{f.name}: loaded, no setup()')
            continue
        before = (len(reg.tools), len(reg.skills), len(reg.commands))
        try: fn(reg)
        except Exception as e:
            reg.notes.append(f'{f.name}: setup() failed ({agent_err(e)})')
            continue
        d = [n - b for n, b in zip((len(reg.tools), len(reg.skills), len(reg.commands)), before)]
        reg.notes.append(f'{f.name}: {d[0]} tool(s), {d[1]} skill(s), {d[2]} command(s)')
    return reg

## Tests


In [ ]:
# User python that adds a tool, a skill and a command -- the escape hatch that means a
# one-off need does not have to become a feature in here.
import tempfile, pathlib
tmp = pathlib.Path(tempfile.mkdtemp())
(tmp/'ext.py').write_text(
    'def setup(reg):\n'
    '    @reg.tool\n'
    '    def shout(text: str) -> str:\n'
    '        "Shout something."\n'
    '        return text.upper()\n'
    '    reg.skill("shouting", "SHOUT MORE", description="how to shout")\n'
    '    reg.command("shout", lambda agent, arg: arg.upper(), help="shout it")\n')
r = Registry()
load(r, paths=[tmp/'ext.py'])
print('tools   :', [t.__name__ for t in r.tools])
print('skills  :', [s.name for s in r.skills])
print('commands:', sorted(r.commands))
print('notes   :', r.notes)
assert [t.__name__ for t in r.tools] == ['shout'] and r.tools[0]('hi') == 'HI'

In [ ]:
# A broken extension is reported, never raised: a typo in a user's file must not be able
# to stop the harness from starting.
(tmp/'bad.py').write_text('this is not python(')
r2 = Registry()
load(r2, paths=[tmp/'bad.py'])
print('notes:', r2.notes)
assert r2.notes and 'failed to load' in r2.notes[0] and not r2.tools

In [ ]:
# An unknown hook name is an error rather than a silent no-op -- a hook that never fires
# because the name was misspelled is the worst of both worlds.
r3 = Registry()
try: r3.on('after_nothing', lambda: None); raise AssertionError('should have raised')
except KeyError as e: print('unknown event ->', e)

# A file with no setup() is left alone, not reported as broken: that is how a shared
# helper module sitting in the same directory should behave.
(tmp/'helper.py').write_text('SHARED = 1\n')
r4 = Registry(); load(r4, paths=[tmp/'helper.py'])
print('notes:', r4.notes)
assert 'no setup()' in r4.notes[0]